### Exercise 3: The Message Window Manager

Topic: Messages & State Management


Goal: Create a custom trimmer that preserves the SystemMessage and ToolCall integrity.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [ ]:
# Import message types from LangChain core for handling different message categories
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage

def trim_history(history, max_messages=2):
    """
    Trim message history while preserving system messages and tool call integrity.
    
    Args:
        history: List of message objects to trim
        max_messages: Maximum number of messages to keep (excluding system message)
    
    Returns:
        Trimmed message list with system message at the beginning if it exists
    """
    # Handle empty history
    if not history:
        return []
    
    # Extract and preserve the system message (should be first if present)
    system_message = None
    rest = history

    # Check if the first message is a system message and separate it
    if isinstance(history[0], SystemMessage):
        system_message = history[0]
        rest = history[1:]  # Keep remaining messages for trimming
    
    # Keep only the last max_messages messages from the remaining history
    trimmed = rest[-max_messages:]
    
    # Ensure tool calls are paired with their responses
    fixed = []
    i = 0
    while i < len(trimmed):
        msg = trimmed[i]
        # Check if this is an AI message with tool calls
        if isinstance(msg, AIMessage) and msg.tool_calls:
            # If the next message is a ToolMessage, keep both together
            if i + 1 < len(trimmed) and isinstance(trimmed[i + 1], ToolMessage):
                fixed.append(msg)  # Add AI message with tool call
                fixed.append(trimmed[i + 1])  # Add corresponding tool response
                i += 2  # Skip both messages
            else:
                # Skip if tool call has no corresponding response
                i += 1
        else:
            # Add non-tool messages as-is
            fixed.append(msg)
            i += 1
    
    # Restore system message at the beginning if it existed
    if system_message:
        return [system_message] + fixed
    return fixed
            

# Create sample message history with various message types
messages = [
    SystemMessage(content="You are a helpful assistant."),  # System instruction
    HumanMessage(content="What is the weather?"),  # User question
    AIMessage(content="", tool_calls=[{"name": "get_weather", "args": {"loc": "NY"}, "id": "1"}]),  # AI tool call
    ToolMessage(content="Sunny, 25C", tool_call_id="1"),  # Tool response
    HumanMessage(content="And in London?")  # Follow-up question
]

# Trim the history to keep only the last 2 messages (plus system message)
trimmed = trim_history(messages, max_messages=2)

# Display the trimmed messages with their types
for msg in trimmed:
    print(type(msg).__name__, msg.content)


SystemMessage You are a helpful assistant.
ToolMessage Sunny, 25C
HumanMessage And in London?
